# Mapping and Visualization

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mihiarc/socialmapper/blob/main/docs/notebooks/05-mapping-visualization.ipynb)

## Learning Objectives

By the end of this notebook, you will be able to:

- Create publication-ready choropleth maps
- Choose appropriate basemaps for your visualization
- Add overlays (boundaries and point markers)
- Display statistics boxes on maps
- Select the right colormap for your data
- Export maps in multiple formats (PNG, PDF, SVG, GeoJSON, Shapefile)

## Prerequisites

- Completed [01-Getting Started](01-getting-started.ipynb)
- Basic understanding of census data from [04-Census Data](04-census-data.ipynb)

## Setup

In [ ]:
# Install SocialMapper from GitHub (latest version)
!pip install -q "socialmapper[routing] @ git+https://github.com/mihiarc/socialmapper.git" folium

# IMPORTANT: After install, go to Runtime > Restart session, then skip this cell

In [ ]:
import os

import socialmapper
print(f"SocialMapper v{socialmapper.__version__}")

from socialmapper import (
    create_isochrone,
    get_census_blocks,
    get_census_data,
    create_map,
    get_poi
)
from IPython.display import Image, display

print("Ready!")

## What is a Choropleth Map?

A **choropleth map** colors geographic areas according to a data variable. Darker/more intense colors typically represent higher values.

Common uses:
- Population density
- Income levels
- Election results
- Health statistics

> **Note:** SocialMapper's `create_map()` function handles the complexity of coordinate reference systems, legends, scale bars, and north arrows automatically.

## Basic Map Creation

Let's create a simple population map.

In [ ]:
# Step 1: Get geographic areas
blocks = get_census_blocks(
    location=(45.5152, -122.6784),  # Portland
    radius_km=3
)
print(f"Found {len(blocks)} census block groups")

# Step 2: Get data for those areas
geoids = [b['geoid'] for b in blocks]
census_result = get_census_data(geoids, variables=["population"])

# Step 3: Combine geometry with data
for block in blocks:
    data = census_result.data.get(block['geoid'], {})
    block['population'] = data.get('population', 0) or 0

# Step 4: Create the map
map_result = create_map(
    data=blocks,
    column="population",
    title="Population by Census Block Group"
)

print(f"\nMap created!")
print(f"Format: {map_result.format}")
print(f"Size: {len(map_result.image_data):,} bytes")

## Displaying Maps in Colab/Jupyter

Use `IPython.display.Image` to render maps directly in your notebook.

> **Tip:** For Colab compatibility, always use `Image(map_result.image_data)` rather than relying on file paths.

In [ ]:
# Display the map inline
display(Image(map_result.image_data))

## Basemap Options

SocialMapper supports multiple basemaps to provide geographic context. The `basemap` parameter controls the background layer.

| Basemap | Description | Best For |
|---------|-------------|----------|
| `'CartoDB.Voyager'` | Clean, light basemap (default) | General use, presentations |
| `'CartoDB.Positron'` | Minimal, grayscale | Print-friendly, formal reports |
| `'CartoDB.DarkMatter'` | Dark theme | Dark mode presentations |
| `'OpenStreetMap.Mapnik'` | Standard OSM | Detailed street context |
| `'Stamen.Toner'` | High-contrast B&W | Artistic, high contrast |
| `'Stamen.TonerLite'` | Light grayscale | Subtle background |
| `'Stamen.Watercolor'` | Watercolor style | Artistic presentations |
| `None` | No basemap (white) | Publication figures |

In [ ]:
# Compare different basemaps
basemaps = [
    ('CartoDB.Voyager', 'Default - Clean & Light'),
    ('CartoDB.Positron', 'Minimal Grayscale'),
    ('CartoDB.DarkMatter', 'Dark Theme'),
    (None, 'No Basemap')
]

for basemap, description in basemaps:
    result = create_map(
        data=blocks,
        column="population",
        title=f"Population ({description})",
        basemap=basemap
    )
    print(f"\n{description}:")
    display(Image(result.image_data))

## Colormap Selection

The `cmap` parameter controls how values are mapped to colors. SocialMapper auto-selects appropriate colormaps, but you can override them.

### Sequential Colormaps (for continuous positive values)
- `'YlGnBu'` (default) - Yellow to Green to Blue
- `'Blues'`, `'Greens'`, `'Reds'`, `'Purples'`
- `'viridis'`, `'plasma'`, `'inferno'`

### Diverging Colormaps (for values with a meaningful center)
- `'RdBu'` - Red to Blue
- `'RdYlGn'` - Red to Yellow to Green
- `'coolwarm'`

### Categorical Colormaps (for distinct categories)
- `'Set3'`, `'Set2'`, `'Paired'`, `'tab10'`

In [ ]:
# Compare different colormaps
colormaps = ['YlGnBu', 'Blues', 'viridis', 'plasma']

for cmap in colormaps:
    result = create_map(
        data=blocks,
        column="population",
        title=f"Population (cmap='{cmap}')",
        cmap=cmap
    )
    print(f"\nColormap: {cmap}")
    display(Image(result.image_data))

## Adding Overlays

Overlays add context to your maps. SocialMapper supports two types:

1. **Boundary overlays** (`overlay_boundary`) - Show isochrone or study area boundaries
2. **Point overlays** (`overlay_points`) - Mark specific locations like POIs or facilities

> **Note:** Overlays are styled automatically - boundaries appear as dashed red lines, and points as red markers.

In [ ]:
# Create an isochrone to use as a boundary overlay
isochrone = create_isochrone(
    location="Portland, OR",
    travel_time=15,
    travel_mode="walk"
)

# Get census blocks within the isochrone
blocks = get_census_blocks(polygon=isochrone)
geoids = [b['geoid'] for b in blocks]
census_result = get_census_data(geoids, variables=["population"])

for block in blocks:
    data = census_result.data.get(block['geoid'], {})
    block['population'] = data.get('population', 0) or 0

print(f"Census blocks: {len(blocks)}")
print(f"Isochrone area: {isochrone['properties']['area_sq_km']:.2f} km²")

In [ ]:
# Map with isochrone boundary overlay
result = create_map(
    data=blocks,
    column="population",
    title="Population with 15-min Walking Boundary",
    overlay_boundary=isochrone  # Pass the isochrone directly
)

print("Map with boundary overlay:")
display(Image(result.image_data))

In [ ]:
# Get POIs for point overlay
libraries = get_poi(
    location="Portland, OR",
    categories=["education"],
    limit=10
)

# Format points for overlay (needs lat, lon, optionally name)
library_points = [
    {'lat': lib['lat'], 'lon': lib['lon'], 'name': lib['name']}
    for lib in libraries
]

print(f"Found {len(library_points)} libraries for overlay")

In [ ]:
# Map with point overlay
result = create_map(
    data=blocks,
    column="population",
    title="Population with Library Locations",
    overlay_points=library_points
)

print("Map with point overlay:")
display(Image(result.image_data))

In [ ]:
# Combine boundary AND point overlays
result = create_map(
    data=blocks,
    column="population",
    title="Population with Walking Boundary & Libraries",
    overlay_boundary=isochrone,
    overlay_points=library_points
)

print("Map with both overlays:")
display(Image(result.image_data))

## Statistics Box

Add a statistics box to display summary information directly on the map. This is useful for presentations and reports.

Two ways to use statistics:

1. **Auto-compute** (`show_stats=True`) - SocialMapper calculates basic stats from your data
2. **Custom stats** (`stats_dict`) - Provide your own key-value pairs

In [ ]:
# Auto-computed statistics
result = create_map(
    data=blocks,
    column="population",
    title="Population with Auto Statistics",
    show_stats=True
)

print("Map with auto-computed statistics:")
display(Image(result.image_data))

In [ ]:
# Custom statistics dictionary
total_pop = sum(b['population'] for b in blocks)
avg_pop = total_pop // len(blocks) if blocks else 0

custom_stats = {
    "Study Area": "Portland, OR",
    "Travel Time": "15-min walk",
    "Block Groups": len(blocks),
    "Total Population": f"{total_pop:,}",
    "Avg Population": f"{avg_pop:,}",
    "Libraries": len(libraries)
}

result = create_map(
    data=blocks,
    column="population",
    title="Library Access Analysis - Portland",
    overlay_boundary=isochrone,
    overlay_points=library_points,
    stats_dict=custom_stats
)

print("Map with custom statistics:")
display(Image(result.image_data))

## Export Formats

SocialMapper supports multiple export formats:

| Format | Parameter | Use Case | Returns |
|--------|-----------|----------|--------|
| PNG | `export_format="png"` | Web, presentations | `image_data` (bytes) |
| PDF | `export_format="pdf"` | Print, publications | `image_data` (bytes) |
| SVG | `export_format="svg"` | Scalable graphics | `image_data` (bytes) |
| GeoJSON | `export_format="geojson"` | Web mapping (Leaflet, Mapbox) | `geojson_data` (dict) |
| Shapefile | `export_format="shapefile"` | GIS software (QGIS, ArcGIS) | `file_path` (str) |

In [ ]:
# PNG (default)
png_result = create_map(
    data=blocks,
    column="population",
    export_format="png"
)
with open("population_map.png", "wb") as f:
    f.write(png_result.image_data)
print(f"PNG saved: population_map.png ({len(png_result.image_data):,} bytes)")

# PDF
pdf_result = create_map(
    data=blocks,
    column="population",
    export_format="pdf"
)
with open("population_map.pdf", "wb") as f:
    f.write(pdf_result.image_data)
print(f"PDF saved: population_map.pdf ({len(pdf_result.image_data):,} bytes)")

# SVG
svg_result = create_map(
    data=blocks,
    column="population",
    export_format="svg"
)
with open("population_map.svg", "wb") as f:
    f.write(svg_result.image_data)
print(f"SVG saved: population_map.svg ({len(svg_result.image_data):,} bytes)")

In [ ]:
import json

# GeoJSON export
geojson_result = create_map(
    data=blocks,
    column="population",
    export_format="geojson"
)

with open("population_data.geojson", "w") as f:
    json.dump(geojson_result.geojson_data, f, indent=2)

print(f"GeoJSON saved: population_data.geojson")
print(f"Features: {len(geojson_result.geojson_data['features'])}")
print("\nThis file can be used with:")
print("  - Leaflet.js")
print("  - Mapbox GL JS")
print("  - QGIS")
print("  - Any GeoJSON-compatible tool")

In [ ]:
# Shapefile export (requires save_path)
shp_result = create_map(
    data=blocks,
    column="population",
    export_format="shapefile",
    save_path="population_shapefile"
)

print(f"Shapefile saved: {shp_result.file_path}")
print("\nThis creates multiple files:")
print("  - .shp (geometry)")
print("  - .shx (index)")
print("  - .dbf (attributes)")
print("  - .prj (projection)")

## Saving Maps Directly

Use `save_path` to write directly to a file during map creation.

In [ ]:
# Save directly during creation
result = create_map(
    data=blocks,
    column="population",
    title="Portland Population Analysis",
    overlay_boundary=isochrone,
    stats_dict=custom_stats,
    save_path="portland_analysis.png"
)

print(f"Saved to: {result.file_path}")

# Display from file
display(Image(filename="portland_analysis.png"))

## Publication-Ready Map Example

Let's create a complete, publication-quality map with all features.

In [ ]:
# Prepare data
isochrone = create_isochrone("Portland, OR", travel_time=20, travel_mode="walk")
blocks = get_census_blocks(polygon=isochrone)

geoids = [b['geoid'] for b in blocks]
census_result = get_census_data(geoids, variables=["population", "median_income"])

# Calculate population density
for block in blocks:
    data = census_result.data.get(block['geoid'], {})
    pop = data.get('population', 0) or 0
    income = data.get('median_income', 0) or 0
    area = block['area_sq_km'] if block['area_sq_km'] > 0 else 0.01
    
    block['population'] = pop
    block['median_income'] = income
    block['density'] = pop / area  # people per km²

# Get libraries for overlay
libraries = get_poi("Portland, OR", categories=["education"], travel_time=20, limit=15)
library_points = [{'lat': lib['lat'], 'lon': lib['lon'], 'name': lib['name']} for lib in libraries]

# Calculate statistics
total_pop = sum(b['population'] for b in blocks)
valid_incomes = [b['median_income'] for b in blocks if b['median_income'] > 0]
avg_income = sum(valid_incomes) // len(valid_incomes) if valid_incomes else 0

stats = {
    "Location": "Portland, OR",
    "Travel Mode": "20-minute walk",
    "Area": f"{isochrone['properties']['area_sq_km']:.1f} km²",
    "Population": f"{total_pop:,}",
    "Avg Income": f"${avg_income:,}",
    "Libraries": len(libraries),
    "Block Groups": len(blocks)
}

In [ ]:
# Create publication-ready density map
density_blocks = [b for b in blocks if b['density'] > 0]

result = create_map(
    data=density_blocks,
    column="density",
    title="Population Density with Library Access - Portland, OR",
    basemap="CartoDB.Positron",
    cmap="YlOrRd",
    overlay_boundary=isochrone,
    overlay_points=library_points,
    stats_dict=stats,
    save_path="portland_publication_map.png"
)

print("Publication-ready map:")
display(Image(result.image_data))

## Interactive Maps with Folium

For interactive exploration, you can use Folium alongside SocialMapper.

In [ ]:
import folium

# Create Folium map
location = (45.5152, -122.6784)  # Portland
m = folium.Map(location=list(location), zoom_start=13)

# Add census blocks colored by population
max_pop = max(b['population'] for b in blocks) or 1

for block in blocks[:50]:  # Limit for performance
    pop = block['population']
    
    # Color scale based on population
    intensity = pop / max_pop
    if intensity > 0.7:
        color = '#d73027'  # Dark red
    elif intensity > 0.4:
        color = '#fc8d59'  # Orange
    elif intensity > 0.2:
        color = '#fee08b'  # Yellow
    else:
        color = '#91cf60'  # Green
    
    folium.GeoJson(
        block['geometry'],
        style_function=lambda x, c=color: {
            'fillColor': c,
            'color': 'black',
            'weight': 1,
            'fillOpacity': 0.6
        },
        tooltip=f"Population: {pop:,}"
    ).add_to(m)

# Add library markers
for lib in libraries:
    folium.Marker(
        location=[lib['lat'], lib['lon']],
        popup=lib['name'],
        icon=folium.Icon(color='blue', icon='book', prefix='fa')
    ).add_to(m)

print(f"Interactive map with {len(libraries)} libraries")
m

## Map Metadata

Every map result includes metadata about the visualization.

In [ ]:
result = create_map(
    data=blocks,
    column="population",
    title="Population Map"
)

print("Map Metadata:")
for key, value in result.metadata.items():
    print(f"  {key}: {value}")

## Troubleshooting

### Common Issues and Solutions

| Issue | Cause | Solution |
|-------|-------|----------|
| Blank map | No data in blocks | Check that `column` exists and has values |
| Map won't display in Colab | Using file path | Use `Image(result.image_data)` instead |
| Colors look wrong | Wrong colormap type | Use sequential for counts, diverging for +/- values |
| Overlay not visible | Wrong data format | Ensure overlay_points has 'lat' and 'lon' keys |
| Stats box overlaps data | Default position | Custom positioning not yet supported |
| Shapefile errors | Missing save_path | Always provide save_path for shapefile export |

### Best Practices

1. **Always filter missing data** before mapping
2. **Use appropriate colormaps** for your data type
3. **Add context** with basemaps and overlays
4. **Include statistics** for publication maps
5. **Export multiple formats** for different uses

In [ ]:
# Best practice: Filter missing data before mapping
valid_blocks = [b for b in blocks if b.get('population', 0) > 0]
print(f"Original blocks: {len(blocks)}")
print(f"Valid blocks: {len(valid_blocks)}")

# Best practice: Verify column exists
column = 'population'
if valid_blocks and column in valid_blocks[0]:
    result = create_map(data=valid_blocks, column=column)
    print(f"Map created successfully for '{column}'")
else:
    print(f"Column '{column}' not found in data")

## Next Steps

Continue with:

- **[Complete Workflow](06-complete-workflow.ipynb)** - Full analysis from start to finish with report generation
- **[Food Desert Case Study](07-food-desert-case-study.ipynb)** - Real-world mapping application